# Neural network architecture across every election, 1997–2024

This extends [notebook 07](07_nn_architecture_learning_rate_study.IPYNB) from three evaluation elections to **1997, 2001, 2005, 2010, 2015, 2017, 2019 and 2024**. See [the interpreted report](nn_architecture_study/REPORT.md) and [complete numerical results](nn_architecture_study/all_elections/RESULTS.md).

Every window uses all available elections through its cutoff for weight updates, the next election for validation, and the following election for evaluation. All four architectures receive ten-seed confirmation, with eight learning rates and patience 10/20/50. No global architecture shortlist is selected using other periods.

**These are retrospective development results, including 2024.** Model fitting, preprocessing, checkpoint selection and local learning-rate selection exclude evaluation outcomes. The supplied feature tables are used as-is; this is not a new audit of their historical availability or boundary mappings. Seat totals refer to the supplied Great Britain rows, not the whole UK Parliament.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "TEST_TRAIN/train.csv").exists())
STUDY = ROOT / "Analysis and model development/nn_architecture_study"
RESULTS = STUDY / "all_elections"
RUN_STUDY = False
# This runner verifies exact equivalence to the reference batching before fitting.
RUNNER = "resume_all_elections.py"
if RUN_STUDY:
    subprocess.run([sys.executable, str(STUDY / RUNNER)], check=True)
configuration = json.loads((RESULTS / "configuration.json").read_text())
detail = pd.read_csv(RESULTS / "election_diagnostics.csv")
party = pd.read_csv(RESULTS / "per_party_recall.csv")
transitions = pd.read_csv(RESULTS / "transitions.csv")
policies = pd.read_csv(RESULTS / "confirmation_policies.csv")
confirmed = pd.read_csv(RESULTS / "confirmation_results.csv")
display(pd.read_csv(RESULTS / "window_coverage.csv").fillna(""))

## Architecture comparisons under two learning-rate policies

`fixed_0.3` holds the learning rate constant. `validation_all_rates` chooses each architecture's rate using its own ten-seed ensemble validation log loss. The latter compares complete tuned procedures: its differences cannot be attributed solely to architecture. Both use patience 20 and restore each seed's best validation checkpoint. A three-seed view is also retained to examine selection stability.

The overall/changed score gives equal weight to overall and changed-seat accuracy, matching the current pipeline selector up to a factor of two. It is a descriptive metric here; rates are still selected by validation log loss.


In [ ]:
for policy in ["fixed_0.3", "validation_all_rates"]:
    print(policy)
    subset = detail.loc[detail.policy == policy]
    display(subset.groupby("architecture").agg(
        mean_accuracy=("accuracy", "mean"), mean_changed_accuracy=("changed_accuracy", "mean"),
        mean_log_loss=("log_loss", "mean"),
        mean_overall_changed_score=("overall_changed_score", "mean"), mean_absolute_seat_error=("seat_count_absolute_error", "mean")))
    display(subset.pivot(index="evaluation_year", columns="architecture", values="accuracy"))
display(Image(filename=str(RESULTS / "architecture_by_election.png")))

## Does the model correctly predict changes?

Changed-seat accuracy asks whether the model names the **correct new party** on seats whose supplied previous-winner label differs from the actual winner. Predicting any change is not sufficient. Compare correct changes against false changes on held seats; the previous-winner baseline has zero correct changes by construction. These are constituency changes, distinct from a change in national government.

Rows without a previous winner remain in overall accuracy but are excluded from change/hold diagnostics. There are 91 such rows in 1997 and one in 2024. The all-row previous-winner baseline treats unavailable predictions as incorrect; compare both models on the known-previous subset for a fair direct comparison.

In [ ]:
display(detail.loc[detail.policy == "validation_all_rates", [
    "evaluation_year", "architecture", "learning_rate", "accuracy", "previous_winner_accuracy",
    "accuracy_gain_pp", "known_previous_accuracy", "previous_winner_known_accuracy",
    "unknown_previous_rows", "unknown_previous_accuracy", "changed_seats", "correct_changes", "changed_accuracy",
    "called_changes", "correct_change_precision", "false_changes_on_held_seats",
    "wrong_destination_on_changed_seats", "held_seat_accuracy"]])

## 1997 and 2010, with 2024 as another government-change case

1997 trains on 1987 and validates on 1992. Raw national poll columns are constant within 1987, so the network cannot learn their cross-election effects from that window; constituency projected shares still vary. The `oth` output exists but has no positive training examples. These are substantial limits on interpreting architecture differences.

2010 trains through 2001 and validates on 2005. This gives more historical variation, but validation is still on a different electoral setting from the evaluation election. The 2010 result led to a Conservative–Liberal Democrat coalition after a hung Parliament, rather than a Conservative overall majority ([UK Parliament](https://www.parliament.uk/about/how/elections-and-voting/general/hung-parliament/)). A seat classifier does not model coalition formation.

`natSW` combines SNP and Plaid Cymru; `oth` is a pooled residual class. A class's predicted seat total uses argmax winners; expected seats sum its marginal probabilities. Neither supplies a parliamentary-majority probability.

The 1997 dataset also lacks previous-election features on 91 rows, which receive training-only imputation. In 2024, two region labels are unseen in training and therefore receive all-zero region indicators under the unchanged encoder. These data limitations are documented in `window_coverage.csv`; they are not corrected as part of this architecture experiment.

In [ ]:
display(Image(filename=str(RESULTS / "government_change_seats.png")))
for year in [1997, 2010, 2024]:
    print(f"Election {year}")
    display(detail.loc[detail.evaluation_year == year])
    display(party.loc[(party.evaluation_year == year) & (party.policy == "validation_all_rates")])
    display(transitions.loc[(transitions.evaluation_year == year) &
        (transitions.policy == "validation_all_rates") & transitions.previous_party.ne(transitions.actual_party)])

## Every election: party seat counts and errors

Aggregate totals can hide errors that cancel between constituencies. Read these alongside all-seat accuracy, changed-seat accuracy, the saved confusion matrices and paired architecture comparisons.

In [ ]:
for year, frame in party.loc[party.policy == "validation_all_rates"].groupby("evaluation_year"):
    print(year)
    seats = frame.pivot(index="party", columns="architecture", values="predicted_seats")
    seats.insert(0, "actual", frame.drop_duplicates("party").set_index("party").actual_seats)
    display(seats)
display(pd.read_csv(RESULTS / "paired_architecture_comparison.csv"))

## Learning rates, patience and seed stability

The best evaluation result among many inspected combinations is a retrospective diagnostic, not a valid selection rule. Rates and checkpoints are chosen locally on the validation election. Seed spread measures optimisation variability; ten seeds are not ten independent future elections.

In [ ]:
display(confirmed.loc[confirmed.patience == 20].groupby(["architecture", "learning_rate"]).agg(
    mean_validation_loss=("validation_log_loss", "mean"), mean_accuracy=("evaluation_accuracy", "mean"),
    mean_log_loss=("evaluation_log_loss", "mean")))
display(policies.loc[policies.policy == "validation_all_rates"].groupby(["architecture", "patience"]).agg(
    mean_accuracy=("evaluation_accuracy", "mean"), mean_changed_accuracy=("evaluation_changed_seat_accuracy", "mean"),
    mean_log_loss=("evaluation_log_loss", "mean")))
display(pd.read_csv(RESULTS / "seed_stability.csv"))
records = pd.read_csv(RESULTS / "confirmation_training_records.csv")
display(records.loc[records.patience == 20].groupby(["train_through", "architecture"]).agg(
    parameters=("parameters", "first"), median_best_epoch=("best_epoch", "median"),
    max_best_epoch=("best_epoch", "max"), capped_runs=("hit_epoch_cap", "sum")))

## Dependence on individual elections and the smaller rate search

Leave-one-election-out averages show whether a descriptive architecture ranking depends on a particular period. Models are not retrained for these summaries. The persistence comparison uses rows with a known previous winner. The second table tests whether the smaller rate set from the original three-election study preserves the full-search selections.


In [ ]:
sensitivity = pd.read_csv(RESULTS / "leave_one_election_out.csv")
display(sensitivity)
rate_comparison = pd.read_csv(RESULTS / "midrange_search_comparison.csv")
display(rate_comparison.loc[(rate_comparison.architecture == "32_16") & (rate_comparison.patience == 20), [
    "evaluation_year", "learning_rate_full", "learning_rate_midrange", "same_rate",
    "midrange_accuracy_delta_pp", "midrange_log_loss_delta"]])


## Reproduction and verification

The full fit is resumable via a data/code/configuration fingerprint. The default notebook reads saved artifacts; set `RUN_STUDY = True` to refit, regenerate diagnostics and audit. The original three-election outputs are preserved in the parent study directory. All new outputs live in `nn_architecture_study/all_elections/`.

The numerical audit verifies election coverage, temporal ordering, data/code hashes for the extension, run counts, checkpoint ordering, validation-only policy selection, source-row alignment, probability validity, reconstructed accuracy/log loss/Brier score and reproduction of the original 24 ten-seed 32/16 validation comparisons.

The accelerated runner checks its batch order, random-generator state, loss histories, checkpoints and probabilities against the reference runner before fitting. Both produce the same numerical experiment. `batch_verification.txt` and `execution_backend.json` record the checks and source hashes; elapsed times mix reference and accelerated runs and are not a controlled architecture-speed comparison.

The resume runner stores checkpoints in the workspace `.nn_study_cache/` by default, so completed trajectories survive Codespace restarts. Override `NN_STUDY_CACHE` to use another location. Run `python "Analysis and model development/nn_architecture_study/resume_all_elections.py"` to fit, summarize and audit in sequence. Do not start a second runner while one is active.


In [ ]:
print((RESULTS / "verification.txt").read_text())
print("Windows:", configuration["windows"])
print("Architectures:", configuration["architectures"])
print("Seeds:", configuration["seeds"])
print("Data hash:", configuration["data_sha256"])
print("2024 data hash:", configuration["evaluation_data_sha256"])